# Tutorial on `xarray.apply_ufunc` function

In this notebook I explore the functioning of function `xarray.apply_ufunc`. My goal is to apply an indicator to an xarray Dataset object cointaining the needed variables with their coordinates and additional coordinates for its optional parameters. In particular, data variables in this object do not depend on the additional coordinates, but the outcome of the computation is an xarray Dataset object with values depending on both the original coordinates and the additional coordinates, which become dimensions.

I do my tests with indicator [heat_wave_frequency](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_frequency). I follow two roads:

1. I do some experiments using `xclim.apply_ufunc`, learning from examples I found on the Internet and in the documentation.
2. I check the source code of indicators to understand how the management of `xarray.DataArray` objects as optional arguments is implemented, since in that case the evaluation is parallelised over the dimensions of those objects. My suspect is that some of the inner operations are broadcasted along the dimensions of the `xarray.DataArray` objects involved and I want to understand if that can be extended easily to my needs (e.g. if function `xarray.apply_ufunc` is used and how).

Along point 2, I start to check [the source code of the indicator](https://github.com/Ouranosinc/xclim/blob/edf1e13b7d47e4ead94506cdbb3bce1f977376f9/xclim/indicators/atmos/_temperature.py#L200). It is created by passing simple metadata, but the core of the implementation is the index used in calculations, which is [`xclim.indices.heat_wave_frequency`](https://github.com/Ouranosinc/xclim/blob/edf1e13b7d47e4ead94506cdbb3bce1f977376f9/xclim/indices/_multivariate.py#L639).
This index works on the multidimensional objects through the function [`compare`](https://github.com/Ouranosinc/xclim/blob/edf1e13b7d47e4ead94506cdbb3bce1f977376f9/xclim/indices/generic.py#L232) from module `xclim.indices.generic`. This function does some checks on its arguments and in the end it uses the functional form of the Python comparison operators (cfr. [`operator`](https://docs.python.org/3/library/operator.html) package), which are retrieved through function [`xarray.core.ops.get_op`](https://github.com/pydata/xarray/blob/cd3ab8d5580eeb3639d38e1e884d2d9838ef6aa1/xarray/core/ops.py#L270).
When the binary operators are applied to `xarray.DataArray` objects, I think their overrided operators are called. In `xarray` some methods from `numpy` are not implemented directly but they are injected (i.e. this is done by [one of the parent classes](https://github.com/pydata/xarray/blob/cd3ab8d5580eeb3639d38e1e884d2d9838ef6aa1/xarray/core/ops.py#L305) of `xarray.DataArray`), however I am not able to figure out where or how the basic comparison operators are implemented. Probably their generation is done automatically with some templates and starts with [this list](https://github.com/pydata/xarray/blob/cd3ab8d5580eeb3639d38e1e884d2d9838ef6aa1/xarray/util/generate_ops.py#L19).
Anyway this exploration is been useful because as long as I delegate the work to xarray functions which act on xclim DataArray objects then I am sure they are parallelised even if I do not know their implementation.

## References

Additional references other than the in-text links are:

- The [official documentation](https://docs.xarray.dev/en/stable/generated/xarray.apply_ufunc.html) is the starting point.
- An [official tutorial](https://tutorial.xarray.dev/advanced/apply_ufunc/apply_ufunc.html) on how to use the method is available.
- The [guide on applying unvectorized functions](https://docs.xarray.dev/en/stable/examples/apply_ufunc_vectorize_1d.html) is useful to extend the vectorization to the dimensions given by the parameters of the indicator.

## Set up

In [ ]:
import xarray  # Version 2024.5.0.
import xclim  # Version 0.49.0.

import utilities

In [ ]:
ds_hourly = utilities.open_dataset()
indicator = xclim.core.indicator.registry["HEAT_WAVE_FREQUENCY"]()

In [ ]:
ds_daily = xarray.Dataset()
_resample = ds_hourly["tas"].resample(time="1D")
ds_daily["tasmin"] = _resample.min()
ds_daily["tasmax"] = _resample.max()